# OpenMP SpMV实验

## 2.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


### 设置 Host 编译器路径

先完成第 01 章的编译器安装，再运行下面的单元格，为当前 Notebook 设置编译器路径。


In [ ]:
import os
from pathlib import Path

# 若安装在其他位置，只修改这一行。
compiler = Path.home() / "opt/BiShengCompiler-3.2.0-aarch64-linux"
os.environ["PATH"] = f"{compiler}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{compiler}/lib:{compiler}/lib/aarch64-unknown-linux-gnu:" + os.environ.get("LD_LIBRARY_PATH", "")


In [ ]:
%%bash
set -e
clang++ --version
cmake --version


<a id="s-02-02_csr_and_spmv_basics"></a>


## 2.2 CSR 与 SpMV 基础

### 本节学习目标

- 理解 row_ptr、col_idx、values
- 追踪一行 SpMV 的计算

### CSR 表示

CSR 用 `row_ptr` 标记每行非零元区间，`col_idx` 给出列索引，`values` 保存数值。第 r 行访问范围是 `[row_ptr[r], row_ptr[r+1])`。

### 手算一个例子

令矩阵为 `[[2,0,3],[0,4,0]]`，向量为 `[1,2,3]`。CSR 表示为：

```text
row_ptr = [0, 2, 3]
col_idx = [0, 2, 1]
values  = [2, 3, 4]
```

第 0 行读取下标 `[0,2)`，得到 `2×1+3×3=11`；第 1 行得到 `4×2=8`。
每行只写自己的一个输出，因此可以把行分给不同线程。长行要做更多乘加，行数相同不一定工作量相同。

### 串行算法

每行独立累加 `values[p] * x[col_idx[p]]`。计算量与 nnz 成正比，但对 x 的间接访问使缓存和带宽成为关键限制。

### 对照源码

串行参考计算位于 `../common/sparse/src/cpu_backend.cpp`，OpenMP 循环位于 `src/openmp_spmv/src/cpu_omp_spmv.cpp`。后者在按行计算的循环前增加并行指令：

```cpp
#pragma omp parallel for schedule(runtime)
for (std::int32_t row = 0; row < matrix.rows; ++row) {
    float sum = 0.0F;
    for (std::int32_t index = matrix.row_ptr[row];
         index < matrix.row_ptr[row + 1]; ++index) {
        sum += matrix.values[index] * x[matrix.col_idx[index]];
    }
    (*y)[row] = sum;
}
```

`src/openmp_spmv/benchmark/spmv_benchmark.cpp` 负责运行两种实现、检查误差和输出时间。


### 课后实践

手工计算一个 3x3 CSR 矩阵的 y，并说明 row_ptr 长度为何为 rows+1。

参考答案见 `answer/02.02_answer.md`。

<a id="s-02-03_openmp_programming_model"></a>


## 2.3 OpenMP 编程模型

### 本节学习目标

- 理解共享内存和线程组（team）
- 分析 static schedule 与线程数

### 共享地址空间

OpenMP 线程共享矩阵和 x，每个循环迭代只写自己的 y 元素。`parallel for` 创建线程组（team）并分配行迭代。

### 调度策略

`schedule(static)` 预先分配连续迭代，开销低；当每行 nnz 差异很大时可能负载不均。dynamic 可改善不均衡，但引入调度开销。

### 设置线程数和调度方式

本实现使用 `schedule(runtime)`，从 `OMP_NUM_THREADS` 读取线程数，从 `OMP_SCHEDULE` 读取调度方式。未设置时默认 16 线程和 static；实验时应根据分配的 CPU 核心数设置线程数。

例如，`OMP_NUM_THREADS=4` 使用 4 个线程，`OMP_SCHEDULE=dynamic,16` 让线程每次领取 16 行。

### 课后实践

比较 static 和 dynamic 对 uniform 与 long-tail 矩阵的可能影响。

参考答案见 `answer/02.03_answer.md`。

<a id="s-02-04_openmp_spmv_implementation"></a>


## 2.4 OpenMP SpMV 实现

### 本节学习目标

- 理解串行与 OpenMP 实现的分工
- 构建并运行 CPU/OpenMP benchmark

### 实验材料

本章使用 CSR 数据结构、串行 SpMV、OpenMP SpMV、测试程序和矩阵生成器。两种 SpMV 使用相同输入，便于检查结果并比较时间。

### 选择输入与线程数

| 矩阵 | 行数与列数 | 目标非零元数 | 行长度特点 |
|---|---:|---:|---|
| U1 / U2 | 10 万 / 100 万 | 100 万 / 1000 万 | 较均匀 |
| L1 / L2 | 10 万 / 100 万 | 100 万 / 1000 万 | 少数长行集中 |
| B1 / B2 | 10 万 / 100 万 | 100 万 / 1000 万 | 块状结构 |

程序先读矩阵缓存，没有时自动生成，无需另行准备文件。首次可用 U1，再用同规模的 L1 比较行长度分布。

在 CANNLab 实例详情查看分配的 CPU/vCPU 数。下面用 1、2、4 线程；资源少于 4 时缩小列表，并将调度对照中的线程数改成相同值。


### 构建方式

CMake 使用 `find_package(OpenMP)` 查找 OpenMP。构建脚本选择鲲鹏毕昇 `clang++`；运行时通过 `OMP_NUM_THREADS` 和 `OMP_SCHEDULE` 设置实验参数。

### 编译工程

下面用鲲鹏毕昇 `clang++` 编译 SpMV 程序。


In [ ]:
%%bash
set -euo pipefail
cd src/openmp_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_COMPILER=clang++
cmake --build build -j


### 运行一次 SpMV

下面使用 U1、4 个线程和 static 调度。`--warmup 1` 先运行一次但不计入统计；`--repeat 3` 再运行三次，输出平均耗时。这里的计算时间不包含 CMake 构建和矩阵生成。

先检查结果误差，再比较串行与并行时间。`relative_error`（部分输出写作 `error`）取逐元素相对误差的最大值：`max_i |y_i-ref_i| / max(|ref_i|, 1e-12)`。结果有限且小于 `1e-6` 时通过检查。


In [ ]:
%%bash
set -euo pipefail
cd src/openmp_spmv
# 学生修改：线程数 OMP_NUM_THREADS、调度方式 OMP_SCHEDULE。
OMP_NUM_THREADS=4 OMP_SCHEDULE=static ./build/bin/spmv_benchmark --matrix U1 --warmup 1 --repeat 3


### 预期现象与结果分析

输出应同时包含 CPU single 与 CPU OpenMP 的时间和相对误差。实际耗时依赖 CPU、编译器、线程绑定和系统负载。

### 课后实践

说明编译命令中的 `-DCMAKE_CXX_COMPILER=clang++` 有什么作用，以及 OpenMP 未被找到时会发生什么。

参考答案见 `answer/02.04_answer.md`。

<a id="s-02-05_scaling_and_bandwidth_analysis"></a>


## 2.5 扩展性与带宽分析

### 本节学习目标

- 计算 speedup 和并行效率
- 解释非线性扩展

### 怎样比较

每次运行都同时测串行与 OpenMP。表中使用该次的 `cpu_single_ms / cpu_openmp_ms` 作为加速比，再除以实际线程数得到并行效率。OpenMP 一线程也有调度开销，加速比不一定等于 1。

`bandwidth_gbps` 是按源码字节模型估算的总带宽；平均每线程带宽再除以 `threads`。该模型包含 values、col_idx、间接读取 x、row_ptr 和输出 y，没有准确模拟缓存命中。效率下降且估算带宽趋平时，可以怀疑访存限制，再用调度对照或 profiler 判断原因。

以 R 行、Z 个非零元计算，源码估算每次访问 `12Z + 4(R+1) + 4R` 字节。U1 取 R=100000、Z=1000000，约为 12.8 MB；若耗时 0.2682 ms，则估算带宽约为 `12.8/0.2682=47.73 GB/s`。

这是按访问次数估算的流量，未扣除缓存命中。再除以线程数得到平均值，并非逐个线程测得的流量。例如加速比 2.8、4 线程时，并行效率为 `2.8/4=70%`。


### 观察顺序

先核对 `Correctness=PASS` 和实际线程数，再比较同一矩阵的时间。若 dynamic 在 L1 上改善、在 U1 上变慢，结合行长度解释负载分配与调度开销。保留没有加速的结果，它同样能说明线程增加的成本。

### 生成本机的扩展性记录

先完成 2.4 的构建。下面固定 U1、static 调度和重复次数，只改变线程数；设备可用核心少于 4 个时，缩小线程数列表。

In [ ]:
%%bash
set -euo pipefail
cd src/openmp_spmv
# 学生修改：下面的线程数列表；比较时保持矩阵和重复次数一致。
for threads in 1 2 4; do
  OMP_NUM_THREADS=$threads OMP_SCHEDULE=static ./build/bin/spmv_benchmark \
    --matrix U1 --warmup 1 --repeat 3 --csv "results/U1_${threads}t_static.csv"
  head -2 "results/U1_${threads}t_static.csv"
done


### 读取结果

每个 CSV 都包含线程数、调度策略、串行和并行耗时、加速比、效率与误差。先检查 error，再比较 speedup 和 efficiency。

### 分析结果

若线程数增加后加速比趋于平缓，结合矩阵结构讨论内存带宽、负载不均和线程调度开销。


### 课后实践

选择两种矩阵结构，比较其并行扩展性并提出基于访问模式的解释。

参考答案见 `answer/02.05_answer.md`。